In [1]:
# Detect Colab and set up the necessary things so it runs
import os
if 'COLAB_GPU' in os.environ:
    !git clone https://github.com/edrosten/squassh.git
    import sys
    sys.path.insert(0, '/content/squassh')
    %pip install pystrict plotly
os.environ["OVERRIDE_UNCLEAN_REPO"]="1"

In [2]:
# importing deep learning packages
import torch
import torch._dynamo

# these are the imports for the data
# you will need to adapt one of these if you want to import your own data
import resi_data
import mark_bates_data

# here we are importing the specific training information, network architecture and device
import train
import network
import device

# this specifies the type of 3D representation that will be used (this will vary between SMLM data and others)
import localisation_data



****************************
Warning, uncommitted changes
****************************




## Load the data. 

If you wish to load Bates data rather than resi, comment the top line and uncommet the one underneath

The `.half()` here converts the data to FP16 (lower than the default 32 bit floats), which takes up less GPU RAM and renders faster.

In [3]:
nupc3d = [t.to(device.device).half() for t in resi_data.load_3d()]
#nupc3d = [t.to(device.device).half() for l in mark_bates_data.load_3d_list() for t in l]

## Set up the training parameters

The rejection parameters is a weighting for how likely the optimisation is to reject a given patch based on quality 
- if you want the model to reject less, make the rejection parameter higher
- if you want the model to reject more, make the rejection parameter lower

In [4]:
rejection = 1.0

Set up the image size used for training and resolution.  `z_scale` is difference in scaling between xy axis and z axis. 

In [5]:
#
data_parameters = train.DataParametersXYYZ(
    image_size_xy = 64,
    image_size_z = 32,
    nm_per_pixel_xy = 3.9,
    z_scale = 2
)

In [6]:
params = train.TrainingParameters()
params.batch_size = 10
params.validity_weight=rejection

### Optimisation parameters

The number of epochs needs to be set so that the system has reached a stable state by the time the optimisation terminates
If you want to check this you would need to load the log file and plot the loss (see below)



The blur reduction schedule below is not the fastest (see train_nupc.py for an optimised one) but is designed to be single step and easy 
to understand. The main limiting factors for speed are the number of points in the model (here 700) and the number of PSF steps taken
Faster optimisation can be achieved by optimising an initial model with fewer points and then re-seeding.

In [7]:
params.schedule[0].epochs = 90
params.schedule[0].initial_psf = 65 # in units of nm
params.schedule[0].final_psf = 33.4 # in units of nm
params.schedule[0].psf_step_every= 30
params.schedule[0].initial_lr= 0.0001
params.schedule[0].final_lr= 0.0001

params.schedule.append(train.TrainingSegment())
params.schedule[1].epochs = 300
params.schedule[1].initial_psf = 24.7
params.schedule[1].final_psf = 13
params.schedule[1].psf_step_every= 100
params.schedule[1].initial_lr= 0.0001
params.schedule[1].final_lr= 0.0001


params.schedule.append(train.TrainingSegment())
params.schedule[2].epochs = 1000
params.schedule[2].initial_psf = 13
params.schedule[2].final_psf = 13
params.schedule[2].psf_step_every= 300
params.schedule[2].initial_lr= 0.0001
params.schedule[2].final_lr= 0.0005


## Create the pytorch dataset

DataSet6Plane defines how the data will be rendered, this was used for all SMLM data.
If you wish to use non-SMLM data you will need to change to a different renderer 

In [8]:
dataset = localisation_data.DataSet6Plane(**vars(data_parameters), data=nupc3d, augmentations=1, device=device.device)

## Set up the network and parameterisation

The helper function `PredictReconstructionStretchExpandValidDan6` sets up a network with 6 plane rendering and with the `AxialStretchRadialExpand` heterogenity. The size of the model (i.e. number of points) is also set here. Note that the dataset is requried since the 6-plane rendering places the planes at the 90th percentile of the data. 

Here we also set up how much the parameterisation can strectch and expand the underlying model.

In [9]:
net, parameterisation =network.PredictReconstructionStretchExpandValidDan6(model_size=700, **vars(data_parameters), data=nupc3d)
parameterisation.max_stretch_factor_axis = 2.0
# Note that with the expansion, the network takes considerably more iterations to converge on the 
# axis. train_nupc.py/ipynb shows how this in a reasnoably short amount of compute time at the 
# expense of a more complex training schedule. 
parameterisation.max_stretch_factor_expand = 1.0

## Perform training

SQUASSH benefits very heavily from `torch.compile()`. This has historically been a little buggy on older versions of torch, and the following commands have been found to help. They may be not be necessary depending on the torch version being used.  

In [10]:
torch._dynamo.config.cache_size_limit=512  # pylint: disable=protected-access
torch.compiler.reset()


Now, send the network to the GPU, compile it and perform the training. The provided training loop sets up the losses as described in the paper, follows the provided learning rate and blur schedule and writes logs.

In [ ]:
net.to(device.device)
fast = torch.compile(net)
train.retrain(fast, dataset, params)

Re-rendering dataset


100%|███████████████████████████████████████████████████████████████████████████████████████████████| 1.22k/1.22k [00:01<00:00, 953it/s]


FWHM = 65.0
122
--------------------------------------------------------------------------------


  0%|                                                                                                           | 0/122 [00:00<?, ?it/s]/home/general/.pyenv/versions/3.11.6/envs/squassh-jupyter-3.11.6/lib/python3.11/site-packages/torch/_inductor/compile_fx.py:140: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(
100%|█████████████████████████████████████████████████████████████████████████████████████████████████| 122/122 [00:42<00:00,  2.88it/s]


6
VALID_ITEMS 466.1372916698456 1218
Done epoch 0 , 38.3% valid
Time per epoch = 49.4s
Estimated remaining = 19h 3m 17s
FWHM = 65.0
122
--------------------------------------------------------------------------------


...

100%|█████████████████████████████████████████████████████████████████████████████████████████████████| 122/122 [00:01<00:00, 85.47it/s]


VALID_ITEMS 813.7869267463684 1218
Done epoch 38 , 66.8% valid
Time per epoch = 1.4s
Estimated remaining = 0h 32m 23s
FWHM = 46.593991028886975
122
--------------------------------------------------------------------------------


100%|█████████████████████████████████████████████████████████████████████████████████████████████████| 122/122 [00:01<00:00, 86.02it/s]


VALID_ITEMS 826.2987394332886 1218
Done epoch 39 , 67.8% valid
Time per epoch = 1.4s
Estimated remaining = 0h 32m 10s
FWHM = 46.593991028886975
122
--------------------------------------------------------------------------------


 96%|█████████████████████████████████████████████████████████████████████████████████████████████    | 117/122 [00:01<00:00, 85.96it/s]

# Analysis

Put the network in eval mode

In [ ]:
net.eval()
None

## Display the underlying structure

The simplest type of analysis is to display the underlying 3D model along with the axis of parameterisation. We provide `make_mesh`, which computes an isosurface of a point cloud by placing a Gaussian at each point weighted by the intensity.

In [ ]:
from save_ply import make_mesh
with torch.no_grad():
    v,f = make_mesh(*net.get_model(), sigma=2.0)


Now show the 3D mesh along with the primary axis 

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio
import sys
pio.renderers.default = 'colab' if 'google.colab' in sys.modules else 'notebook'

fig = go.Figure(go.Mesh3d(
    x=v[:,0], y=v[:,1], z=v[:,2],
    i=f[:,0], j=f[:,1], k=f[:,2]
))
axis_points = parameterisation.get_axis().cpu().detach()
axis_points = torch.stack([-axis_points, axis_points], 0)*100
fig.add_scatter3d(x=axis_points[:,0], y=axis_points[:,1], z=axis_points[:,2], mode='lines')
fig.show()

## Plot the loss against from the log file

The log file is a simple text based format, designed to be parsed using the kind of technique shown below. 

In [ ]:
from git import dirname
from pathlib import Path
log_file = Path('log')/dirname/"loss.txt"
loss_data=[]
with log_file.open("r") as file:
    for line in file:
        if "ITERATION" in line:
            loss_data.append([float(i) for i in line.split()[0:2]])        
loss_data = torch.tensor(loss_data)

Now we plot the loss over time. Each jump in loss is caused when the PSF of the rendering during training steps down, at a rate determeined by the `psf_step_every` parameter above. Loss data is per batch and so is very noisy, so it is alos useful to smooth it for display.

In [ ]:
import matplotlib.pyplot as plt
import scipy
smoothed = scipy.signal.lfilter(*scipy.signal.cheby2(6, 40, 0.01), loss_data[:,1])

plt.plot(loss_data[:,0], smoothed)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

## More complex analysis

For more complex analyses, we need to process all the data one more time with the network and collect information, such as the which ring appears with high (NR) or low (CR) Z. Usiung a batch size of 1 is slow, but simplifies things and we only need one pass.

In [ ]:
from torch.utils.data import DataLoader
from tqdm import tqdm
from localisation_data import fwhm_to_sigma

loader = DataLoader(dataset, batch_size=1, shuffle=False)

# We need to pass the rendering sigma into the network. This needs to 
# be as a tensor on the correct device
fwhm_t = torch.tensor(13.0).to(device.device)

scales = []
axes = []

net.eval()
with torch.no_grad():
    for batch in tqdm(loader):
        # net() runs the network and produces a rendered image output. We want to analyze the 
        # results, not just images, so we use process_input which outputs all the intermediate
        # data including logits for the parameterisation
        t,r,_,is_valid,parameters = net.process_input(batch, min_sigma_nm=fwhm_to_sigma(fwhm_t))

        # Compute various things from the parameterisation
        scale = parameterisation.compute_scale_from_parameters(parameters).cpu().item()
        expand = parameterisation.compute_expand_from_parameters(parameters).cpu().item()
        
        # Reject invalid data from the network. Also, a number of NPCs have only one visible ring 
        # which the network will reproduce by outputting strong in-axis squash in an attempt to 
        # merge the two rings together. These NPCs are not useful to analyze.
        if is_valid > 0.5 and scale > 0.8:
            scales.append([scale, expand])
            
            # Also record the direction of the axis as it appears in image space. This is so we
            # can identify which ring has positive Z and which has negative Z
            axes.append((r @ parameterisation.get_axis().unsqueeze(1)).squeeze())
        

scales = torch.tensor(scales)
axes = torch.stack(axes, 0)

        
        

### Analysis of the Z scaling

In [ ]:
plt.hist(scales[:,0], 30)
plt.xlabel('Relative scaling of the underlying model in Z')
plt.ylabel('Count')

### Display the NR and CR rings

First, we need to determine which of the rings is NR and which is CR. NR has more positive Z and CR has more negative Z, in imaging space. The underling model has a random orientation, but we know which direction the axis points, so work out whether it points to the +Z or -Z ring.

In [ ]:
axis_sign = axes[:,2].cpu().mean().sign()

Now construct a matrix which rotates the model into image space with Z upwards. 

In [ ]:
import matrix
# This constructs a matrix with the given axis aligned with Z.
R = matrix.so3_6D(torch.cat([axis_sign * parameterisation.get_axis().detach().cpu(), torch.ones(3)]).unsqueeze(0)).squeeze(0)
# "Swap" X and Z, such that new Z has the same sign as old X
R = (matrix.euler(torch.tensor([-torch.pi/2]), 'y') @ R).squeeze(0)

Now find the average model.

In [ ]:
average_scale, average_expand = scales.mean(0)
S = matrix.scale_along_axis_and_expand_matrix(parameterisation.get_axis().cpu(), average_scale.unsqueeze(0), average_expand.unsqueeze(0)).squeeze(0)
average_model = net.get_model()[0].cpu() @ S @ R.permute(1,0) # S is symmetric of course.

Now threshold Z to find the NR (top) and CR (bottom) rings.

In [ ]:
nr_mask = average_model[:,2] > 0
cr_mask = nr_mask.logical_not()

And render using the model weights. A bit of care is needed to ensure that there is a batch dimension and all tensors are on the same device. We pick the GPU since it's faster, though it doesn't matter much here.

In [ ]:
import render
pts_nr = average_model[nr_mask,0:2].to(device.device).unsqueeze(0)
pts_cr = average_model[cr_mask,0:2].to(device.device).unsqueeze(0)

sigma = fwhm_to_sigma(torch.tensor([7])).to(device.device)
with torch.no_grad():
    nr_img = render.render_batch_weights(pts_nr, sigma, net.get_model()[1][nr_mask].unsqueeze(0), 0.5, 256).squeeze().cpu()
    cr_img = render.render_batch_weights(pts_cr, sigma, net.get_model()[1][cr_mask].unsqueeze(0), 0.5, 256).squeeze().cpu()


plt.subplot(1,2,1)
plt.imshow(nr_img, cmap='hot')
plt.title('NR')
plt.subplot(1,2,2)
plt.imshow(cr_img, cmap='hot')
plt.title('CR')
plt.show()